# Day 6 — SQL: Window Functions Part 1

> **Topics:** `ROW_NUMBER` · `RANK` · `DENSE_RANK` · `PARTITION BY` · `ORDER BY`  
> **Run order:** top to bottom — Cell 2 connects and creates all tables inline

In [ ]:
%load_ext sql

USERNAME = 'postgres'
PASSWORD = 'hariom'
HOST     = 'localhost'
PORT     = '5432'
DBNAME   = 'postgres'

%sql postgresql://{USERNAME}:{PASSWORD}@{HOST}:{PORT}/{DBNAME}

In [ ]:
%%sql
DROP TABLE IF EXISTS d6w_sales CASCADE;

CREATE TABLE d6w_sales (
    sale_id  SERIAL PRIMARY KEY,
    emp_id   INTEGER,
    emp_name VARCHAR(20),
    region   VARCHAR(10),
    month    INTEGER,
    revenue  NUMERIC(10, 2)
);

INSERT INTO d6w_sales (emp_id, emp_name, region, month, revenue) VALUES
  (1, 'Alice',  'North',  1, 5000.00),
  (2, 'Bob',    'North',  1, 4200.00),
  (3, 'Carol',  'North',  1, 4200.00),   -- tie with Bob
  (4, 'Dave',   'North',  1, 3800.00),
  (5, 'Eve',    'South',  1, 6100.00),
  (6, 'Frank',  'South',  1, 5500.00),
  (7, 'Grace',  'South',  1, 5500.00),   -- tie with Frank
  (8, 'Hank',   'South',  1, 4000.00),
  (1, 'Alice',  'North',  2, 5300.00),
  (2, 'Bob',    'North',  2, 4800.00),
  (3, 'Carol',  'North',  2, 4000.00),
  (5, 'Eve',    'South',  2, 5900.00),
  (6, 'Frank',  'South',  2, 6300.00),   -- Frank tops South in month 2
  (7, 'Grace',  'South',  2, 5100.00);

SELECT 'Tables created' AS status;

In [ ]:
%%sql
SELECT * FROM d6w_sales ORDER BY region, month, revenue DESC;

---
## 1. ROW_NUMBER — Always Unique

Even tied rows get different numbers. Useful for deduplication (pick row #1 per group).

In [ ]:
%%sql
SELECT emp_id, emp_name, region, month, revenue,
       ROW_NUMBER() OVER (PARTITION BY region, month ORDER BY revenue DESC) AS rn
FROM   d6w_sales
ORDER  BY region, month, rn;
-- Bob and Carol both have 4200 in North/month1
-- ROW_NUMBER gives one of them 2 and the other 3 — no ties

---
## 2. RANK — Same Rank for Ties, Then Gap

In [ ]:
%%sql
SELECT emp_id, emp_name, region, month, revenue,
       RANK() OVER (PARTITION BY region, month ORDER BY revenue DESC) AS rnk
FROM   d6w_sales
ORDER  BY region, month, rnk;
-- Bob and Carol: both get rank 2
-- Dave gets rank 4 (rank 3 is skipped — that's the gap)

---
## 3. DENSE_RANK — Same Rank for Ties, No Gap

In [ ]:
%%sql
SELECT emp_id, emp_name, region, month, revenue,
       DENSE_RANK() OVER (PARTITION BY region, month ORDER BY revenue DESC) AS drnk
FROM   d6w_sales
ORDER  BY region, month, drnk;
-- Bob and Carol: both get dense_rank 2
-- Dave gets dense_rank 3 (no gap)

---
## 4. All Three Side by Side — See the Difference

In [ ]:
%%sql
-- Filter to North/month1 to see the tie rows clearly
SELECT emp_name, revenue,
       ROW_NUMBER() OVER (PARTITION BY region, month ORDER BY revenue DESC) AS rn,
       RANK()       OVER (PARTITION BY region, month ORDER BY revenue DESC) AS rnk,
       DENSE_RANK() OVER (PARTITION BY region, month ORDER BY revenue DESC) AS drnk
FROM   d6w_sales
WHERE  region = 'North' AND month = 1
ORDER  BY revenue DESC;
-- Alice:        rn=1  rnk=1  drnk=1
-- Bob (4200):   rn=2  rnk=2  drnk=2
-- Carol (4200): rn=3  rnk=2  drnk=2
-- Dave:         rn=4  rnk=4  drnk=3  ← RANK skips 3, DENSE_RANK doesn't

---
## 5. Top-N per Group — CTE + Filter

You cannot use window functions in `WHERE` directly. Wrap in a CTE first.

In [ ]:
%%sql
-- Top 2 employees by revenue per region (month 1) — ties included with DENSE_RANK
WITH ranked AS (
    SELECT emp_id, emp_name, region, month, revenue,
           DENSE_RANK() OVER (PARTITION BY region, month ORDER BY revenue DESC) AS drnk
    FROM   d6w_sales
    WHERE  month = 1
)
SELECT * FROM ranked WHERE drnk <= 2
ORDER  BY region, drnk;
-- North: Alice(1), Bob(2), Carol(2) — 3 rows because of tie at rank 2
-- South: Eve(1), Frank(2), Grace(2) — 3 rows because of tie at rank 2

---
## 6. Window Without PARTITION BY — Entire Table

In [ ]:
%%sql
-- No PARTITION BY → rank globally across all regions and months
SELECT emp_name, region, month, revenue,
       DENSE_RANK() OVER (ORDER BY revenue DESC) AS global_rank
FROM   d6w_sales
ORDER  BY global_rank
LIMIT  5;

---
## Quick Reference

| Function | Ties | Gap | Use for |
|----------|------|-----|---------|
| `ROW_NUMBER()` | No | — | Dedup, pick one per group |
| `RANK()` | Yes | Yes | Olympic-style ranking |
| `DENSE_RANK()` | Yes | No | Top-N filtering |

```sql
-- Template
FUNCTION() OVER (PARTITION BY col ORDER BY val DESC)

-- Top-N
WITH cte AS (SELECT *, DENSE_RANK() OVER (PARTITION BY grp ORDER BY val DESC) AS drnk FROM t)
SELECT * FROM cte WHERE drnk <= N;
```